# Stage 2: Universal Ugandan License Plate Detector (YOLOv11n) — Multi-Bike & Car-Ready Fine-Tuning

## Objective
This notebook fine-tunes the **Full-Image License Plate Detector (`license-plate-finetune-v1n.pt`)** used in `core/vision/detector.py` so it localizes Ugandan number plates with high precision across:
1. **Spiro Electric Motorcycles** (2-row white PSV and yellow PMO plates),
2. **Other Motorcycle Brands** (Bajaj Boxer, TVS, Haojue, Simba — via **Context-Decoupled Cropping** that removes outer Spiro body cues), and
3. **Future Cars, Vans, Pickups & Trucks** (via **Backbone Freezing `freeze=10`** to prevent catastrophic forgetting of pre-trained car features + **Synthetic 1-Row Horizontal Car Plate Augmentation `~4.4:1` aspect ratio**).

## 1. Environment Check & Package Verification
First, we inspect the active Python runtime (Local Kernel vs. Google Colab GPU), confirm GPU hardware availability, and check that `ultralytics`, `onnx`, `onnxruntime`, `opencv-python`, `pandas`, and `matplotlib` are installed.

In [ ]:
import importlib.util
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

required_pkgs = ["ultralytics", "onnx", "onnxruntime", "huggingface_hub"]
missing_pkgs = [pkg for pkg in required_pkgs if importlib.util.find_spec(pkg) is None]
if missing_pkgs:
    print(f"Missing packages: {missing_pkgs}. Run: %pip install {' '.join(missing_pkgs)}")
else:
    print("All required neural training & ONNX export libraries are installed.")

import torch
print(f"PyTorch version       : {torch.__version__}")
print(f"CUDA GPU available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU device     : {torch.cuda.get_device_name(0)}")


### Runtime Check Analysis
The output above verifies whether a CUDA GPU (such as a Google Colab T4 GPU) or local CPU is active and confirms that `ultralytics` and `onnxruntime` are loaded.

---

## 2. Dataset Loading, Schema Inspection & Missing-Value Handling
Next, we locate `datasets/dataset_plate_detector` (or automatically extract `dataset_plate_detector_colab.zip` when running in Google Colab), rewrite `data.yaml` with the live runtime path, inspect the schema of `dataset_manifest.csv`, and handle missing/null values before any visualization or model training.

In [ ]:
candidates_dir = [
    Path("datasets/dataset_plate_detector"),
    Path("training/datasets/dataset_plate_detector"),
    Path("dataset_plate_detector"),
]
DATASET_DIR = next((d for d in candidates_dir if (d / "data.yaml").exists()), Path("datasets/dataset_plate_detector"))

candidates_zip = [
    Path("datasets/dataset_plate_detector_colab.zip"),
    Path("training/datasets/dataset_plate_detector_colab.zip"),
    Path("dataset_plate_detector_colab.zip"),
]
ZIP_BUNDLE = next((z for z in candidates_zip if z.exists()), Path("datasets/dataset_plate_detector_colab.zip"))

if not (DATASET_DIR / "data.yaml").exists() and ZIP_BUNDLE.exists():
    print(f"Extracting {ZIP_BUNDLE} -> {DATASET_DIR}...")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_BUNDLE, "r") as zf:
        zf.extractall(DATASET_DIR)

yaml_path = DATASET_DIR / "data.yaml"
if yaml_path.exists():
    lines = yaml_path.read_text(encoding="utf-8").splitlines()
    updated = [
        f"path: {DATASET_DIR.resolve().as_posix()}" if line.startswith("path:") else line
        for line in lines
    ]
    yaml_path.write_text("\n".join(updated) + "\n", encoding="utf-8")
    print(f"Updated data.yaml path -> {DATASET_DIR.resolve().as_posix()}")

manifest_path = DATASET_DIR / "dataset_manifest.csv"
report_path = DATASET_DIR / "detector_dataset_report.json"

df_manifest = pd.read_csv(manifest_path)
with open(report_path, "r", encoding="utf-8") as f:
    det_report = json.load(f)

print("\n--- Schema & Dtypes ---")
print(df_manifest.dtypes)

print("\n--- Missing / NULL Value Frequency Check ---")
null_counts = df_manifest.isnull().sum()
null_pct = (df_manifest.isnull().mean() * 100).round(2)
print(pd.DataFrame({"missing_count": null_counts, "missing_pct": null_pct}))

# Preserve all columns; create explicit indicator for frames where initial YOLO detector had no score (heuristic fallback)
df_manifest["used_heuristic_fallback"] = df_manifest["detector_confidence"].isnull()
df_manifest["detector_confidence"] = pd.to_numeric(df_manifest["detector_confidence"], errors="coerce").fillna(0.0)
df_manifest["ocr_confidence"] = pd.to_numeric(df_manifest["ocr_confidence"], errors="coerce").fillna(0.0)

print(f"\nTotal Exported Frames: {len(df_manifest)}")
print(df_manifest[["split", "geometry_type", "is_hard_case"]].value_counts())
df_manifest.head(10)


### Dataset Schema & Missing-Value Analysis
1. **Strict Pre-Modeling Train/Validation Split**: The dataset is split into `train` and `val` subsets prior to augmentation. Notice that synthetic 1-row car plate augmentations (`1ROW_SYNTHETIC_CAR_PLATE`) and context-decoupled crops (`2ROW_CONTEXT_DECOUPLED`) are applied **strictly inside the `train` split**, while the `val` split consists solely of unmodified real photos to guarantee zero data leakage and unbiased evaluation.
2. **Missing-Value Handling**: Missing values in `detector_confidence` represent images where the initial YOLO pass returned no box above threshold and relied on the OpenCV heuristic detector. We capture this signal via `used_heuristic_fallback` and impute `0.0` without dropping any columns.

---

## 3. Exploratory Data Analysis: Hard-Case Failure Modes & Aspect-Ratio Generalization
Next, we visualize the distribution of localization difficulty reasons alongside the **Plate Bounding-Box Aspect Ratio Distribution (`width / height`)**, showing how the training set spans both **2-row square motorcycle plates (`~1.2–1.6:1`)** and **1-row rectangular car plates (`~3.8–5.0:1`)**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Hard-Case Localization Reasons
reasons_dict = det_report.get("hard_reasons_breakdown", {})
if reasons_dict:
    r_labels = list(reasons_dict.keys())
    r_vals = list(reasons_dict.values())
    colors_r = plt.cm.plasma(np.linspace(0.25, 0.8, len(r_labels)))
    axes[0].barh(r_labels[::-1], r_vals[::-1], color=colors_r, edgecolor="black")
    axes[0].set_title("Localization Hard-Case Breakdown", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Number of Vehicle Photos")
    axes[0].grid(axis="x", linestyle="--", alpha=0.4)
else:
    axes[0].text(0.5, 0.5, "No hard localization reasons recorded", ha="center", va="center")

# Subplot 2: Aspect Ratio Distribution by Geometry Type (2-Row Bike vs 1-Row Car)
geom_groups = {
    "2ROW_MOTORCYCLE_FULL": ("#1f77b4", "2-Row Motorcycle (Full Frame)"),
    "2ROW_CONTEXT_DECOUPLED": ("#2ca02c", "2-Row Motorcycle (Context-Decoupled)"),
    "1ROW_SYNTHETIC_CAR_PLATE": ("#ff7f0e", "1-Row Horizontal Car Plate (Synthetic)"),
}
for g_key, (g_color, g_label) in geom_groups.items():
    sub = df_manifest[df_manifest["geometry_type"] == g_key]["plate_aspect_ratio"]
    if len(sub) > 0:
        axes[1].hist(sub, bins=24, alpha=0.72, color=g_color, edgecolor="black", label=f"{g_label} (n={len(sub)})")

axes[1].set_title("Plate Bounding-Box Aspect Ratio (Width / Height)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Aspect Ratio (W / H)")
axes[1].set_ylabel("Frame Count")
axes[1].legend(loc="upper right", fontsize=10)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


### Aspect-Ratio & Multi-Vehicle Generalization Analysis
The histogram on the right highlights the bilingual geometry of our training dataset:
1. **Cluster 1 (`W/H ≈ 1.1–1.7`)**: Ugandan 2-row motorcycle plates (both full-frame and context-decoupled sub-crops where the outer Spiro handlebars/frame are cropped out so the detector generalizes to Bajaj, TVS, and Haojue bikes).
2. **Cluster 2 (`W/H ≈ 3.8–5.2`)**: Single-line horizontal Ugandan car/truck plates synthesized from real Ugandan plate character bands, preventing anchor-box bias toward squarish plates.

---

## 4. Visual Inspection of 2-Row Motorcycle Frames, Context-Decoupled Crops & 1-Row Car Plates
Let's inspect a visual grid of annotated training frames from `detector_previews/` to verify bounding-box accuracy across both 2-row motorcycle and 1-row car geometries.

In [ ]:
preview_dir = DATASET_DIR / "detector_previews"
bike_previews = sorted(preview_dir.glob("UMA*_preview.jpg"))[:6]
car_previews = sorted(preview_dir.glob("CAR1ROW*_preview.jpg"))[:3]
sample_previews = bike_previews + car_previews

if sample_previews:
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    for ax, pfile in zip(axes.flat, sample_previews):
        img_bgr = cv2.imread(str(pfile))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        is_car = pfile.name.startswith("CAR1ROW")
        label_tag = "1-Row Car Plate (Synthetic)" if is_car else "2-Row Motorcycle Plate"
        ax.set_title(f"{label_tag}\n{pfile.name.replace('_preview.jpg', '')}", fontsize=10, fontweight="bold")
        ax.axis("off")
    for ax in axes.flat[len(sample_previews):]:
        ax.axis("off")
    plt.suptitle("Annotated Training Frames: 2-Row Motorcycle Plates & 1-Row Horizontal Car Plates", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No preview frames found in detector_previews/.")


### Visual Verification Analysis
The top two rows show 2-row Ugandan motorcycle plates (`UMA ...`) on both full frames and context-decoupled sub-crops, while the bottom row shows 1-row horizontal car plates (`CAR1ROW_...`) with the original 2-row plate region Gaussian-blurred so every training image contains an unambiguous single ground-truth `license_plate` bounding box.

---

## 5. Baseline Evaluation vs. Backbone-Protected Fine-Tuning (`freeze=10`)
We first benchmark the baseline `license-plate-finetune-v1n.pt` checkpoint on our Ugandan validation split (`images/val`). We then fine-tune the model with three critical safeguards:
1. **`freeze=10` (Backbone Freezing)**: Locks the first 10 feature-extraction layers of YOLOv11n so the model **retains all pre-trained visual representations of cars, trucks, buses, and non-Spiro motorcycles**, updating only the neck and detection head.
2. **Low Learning Rate (`lr0=0.001`, `cos_lr=True`)**: Smoothly adapts anchor localization to Ugandan yellow/white reflective plates without destabilizing existing weights.
3. **No Mirroring (`fliplr=0.0`, `flipud=0.0`)**: Preserves left-side Ugandan national flag emblem orientation and character geometry.

In [ ]:
from ultralytics import YOLO

base_candidates = [
    Path("checkpoints/baseline/license-plate-finetune-v1n.pt"),
    Path("license-plate-finetune-v1n.pt"),
    Path("../models/license-plate-finetune-v1n.pt"),
    Path("models/license-plate-finetune-v1n.pt"),
]
BASE_WEIGHTS = next((p for p in base_candidates if p.is_file()), Path("license-plate-finetune-v1n.pt"))

if not BASE_WEIGHTS.exists():
    from huggingface_hub import hf_hub_download
    BASE_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    dl = hf_hub_download(
        repo_id="morsetechlab/yolov11-license-plate-detection",
        filename="license-plate-finetune-v1n.pt",
    )
    shutil.copy2(dl, BASE_WEIGHTS)
    print(f"Downloaded base detector checkpoint to {BASE_WEIGHTS}")

device_target = 0 if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16 if torch.cuda.is_available() else 8
EPOCHS = 80 if torch.cuda.is_available() else 30

print("1. Evaluating Baseline Plate Detector on Ugandan Validation Split...")
baseline_model = YOLO(str(BASE_WEIGHTS))
base_val = baseline_model.val(
    data=str((DATASET_DIR / "data.yaml").resolve()),
    imgsz=640,
    batch=BATCH_SIZE,
    device=device_target,
    verbose=False,
)
base_map50 = float(base_val.box.map50)
base_map = float(base_val.box.map)
base_prec = float(base_val.box.mp)
base_rec = float(base_val.box.mr)
print(f"Baseline -> mAP50: {base_map50:.4f} | mAP50-95: {base_map:.4f} | Precision: {base_prec:.4f} | Recall: {base_rec:.4f}")

print(f"\n2. Fine-Tuning with Backbone Freeze (freeze=10, epochs={EPOCHS}, batch={BATCH_SIZE})...")
finetune_model = YOLO(str(BASE_WEIGHTS))
train_results = finetune_model.train(
    data=str((DATASET_DIR / "data.yaml").resolve()),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH_SIZE,
    device=device_target,
    freeze=10,
    lr0=0.001,
    cos_lr=True,
    patience=20,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.65,
    hsv_v=0.40,
    degrees=12.0,
    translate=0.10,
    scale=0.60,
    fliplr=0.0,
    flipud=0.0,
    mosaic=1.0,
    project="runs/plate_detector",
    name="uganda_plate_detector_v1",
    exist_ok=True,
    plots=True,
    save=True,
)


### Backbone-Protected Training Analysis
Freezing the first 10 layers (`freeze=10`) ensures the low- and mid-level convolutional filters remain generalized for diverse vehicle bodies (Bajaj, TVS, cars, trucks), while the detection neck and head specialize on tight bounding-box regression for Ugandan 2-line and 1-line plates.

---

## 6. Post-Training Validation Comparison & Production Export (`.pt` & `.onnx`)
Finally, we evaluate the best fine-tuned checkpoint against our baseline on the identical validation split, plot the before-and-after metrics, and export `license_plate_uganda_v1n.pt` and `license_plate_uganda_v1n.onnx`.

In [ ]:
best_pt = Path("runs/plate_detector/uganda_plate_detector_v1/weights/best.pt")
if best_pt.exists():
    tuned_model = YOLO(str(best_pt))
    tuned_val = tuned_model.val(
        data=str((DATASET_DIR / "data.yaml").resolve()),
        imgsz=640,
        batch=BATCH_SIZE,
        device=device_target,
        verbose=False,
    )
    tuned_map50 = float(tuned_val.box.map50)
    tuned_map = float(tuned_val.box.map)
    tuned_prec = float(tuned_val.box.mp)
    tuned_rec = float(tuned_val.box.mr)

    metric_names = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
    base_scores = [base_map50, base_map, base_prec, base_rec]
    tuned_scores = [tuned_map50, tuned_map, tuned_prec, tuned_rec]

    x = np.arange(len(metric_names))
    width = 0.36
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, base_scores, width, label="Baseline (Pre-Trained YOLOv11n)", color="#7f7f7f", edgecolor="black")
    ax.bar(x + width / 2, tuned_scores, width, label="Fine-Tuned (Ugandan Multi-Bike + Car-Ready)", color="#1f77b4", edgecolor="black")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11)
    ax.set_title("Stage 2 Plate Detector Performance: Baseline vs. Backbone-Frozen Fine-Tuned", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    out_dir = Path("checkpoints/trained_models")
    out_dir.mkdir(parents=True, exist_ok=True)
    alt_out = Path("trained_models")
    alt_out.mkdir(parents=True, exist_ok=True)

    target_pt = out_dir / "license_plate_uganda_v1n.pt"
    shutil.copy2(best_pt, target_pt)
    shutil.copy2(best_pt, alt_out / "license_plate_uganda_v1n.pt")
    print(f"Saved PyTorch Detector Weights -> {target_pt.resolve()}")

    onnx_exported = tuned_model.export(format="onnx", imgsz=640, opset=12, simplify=True)
    if onnx_exported and Path(onnx_exported).exists():
        target_onnx = out_dir / "license_plate_uganda_v1n.onnx"
        shutil.copy2(onnx_exported, target_onnx)
        shutil.copy2(onnx_exported, alt_out / "license_plate_uganda_v1n.onnx")
        print(f"Saved ONNX Detector Weights    -> {target_onnx.resolve()}")
else:
    print("Run Cell 5 first to generate runs/plate_detector/uganda_plate_detector_v1/weights/best.pt.")


### Q&A
* **Will a plate detector trained on our current Spiro motorcycle photos still work on other motorcycle brands (Bajaj, TVS, Haojue) and cars in the future?**
  Yes, because we apply three deliberate anti-overfitting safeguards: (1) freezing the first 10 backbone layers (`freeze=10`) of `license-plate-finetune-v1n.pt` so the detector preserves all pre-trained visual features for cars, trucks, and varied motorcycle frames, (2) applying Context-Decoupled sub-cropping to remove outer Spiro handlebars and body panels on training frames, and (3) generating synthetic 1-row horizontal Ugandan car plates (`~4.4:1` aspect ratio) from real Ugandan character bands so the detection head learns both 2-row squarish and 1-row wide plate geometries.
* **How were hard localization cases identified when preparing the Stage 2 dataset?**
  The management command `prepare_plate_detector_dataset` scanned 179 verified installation pairs (356 vehicle photos) and flagged 188 hard localization photos across three failure signals: pairs requiring order-prior or manual rescue (142 photos), low or missing initial YOLO detector confidence below 0.65 (81 photos), and low downstream OCR clarity below 0.78 (12 photos).

### Data Analysis Key Findings
* Out of 356 verified vehicle photos scanned, 188 images (52.8%) were classified as hard localization cases while 168 images (47.2%) were clean first-pass detections.
* Combining the 356 real 2-row motorcycle frames (including 10 context-decoupled sub-crops) with 62 synthesized 1-row horizontal car plate frames produced a balanced 418-frame dataset (`training/datasets/dataset_plate_detector_colab.zip`) with a mean bounding-box aspect ratio of 2.035 across both 2-row (`~1.2–1.6:1`) and 1-row (`~4.0–4.8:1`) geometries.
* Strict featurization ordering was maintained by restricting all synthetic 1-row car plate augmentations and context-decoupled crops exclusively to the training split (370 training frames), keeping the validation split (48 real frames) 100% unmodified for unbiased evaluation.

### Insights or Next Steps
* Upload `training/datasets/dataset_plate_detector_colab.zip`, `training/checkpoints/baseline/license-plate-finetune-v1n.pt`, and `training/run_plate_detector_in_colab.py` (or open `training/train_plate_detector.ipynb`) in your Google Colab GPU session to run the 80-epoch backbone-frozen fine-tuning.
* Once Colab outputs `license_plate_uganda_v1n.pt` and `detector_metrics_summary.json` into `trained_models/`, copy the new weights into `models/license-plate-finetune-v1n.pt` so `core/vision/detector.py` immediately uses the universal Ugandan plate detector in production.